## CuPy

### Table of Contents
1. [Creating Arrays: CPU vs. GPU](#1.-Creating-Arrays:-CPU-vs.-GPU)
2. [Basic Operations](#2.-Basic-Operations)
   - [Sequential Operations & Memory](#Sequential-Operations-&-Memory)
3. [Complex Operations (Linear Algebra)](#3.-Complex-Operations-(Linear-Algebra))
   - [Agnostic Code (NumPy Dispatch)](#Agnostic-Code-(NumPy-Dispatch))
4. [Device Management](#4.-Device-Management)
5. [Exercise - NumPy to CuPy](#Exercise---NumPy-to-CuPy)
   - [Part 1](#Part-1)
   - [Part 2](#Part-2)

---

Let's shift gears to high-level array functionality using **[CuPy](https://cupy.dev/)**.

#### What is CuPy?
CuPy is a library that implements the familiar **NumPy API** but runs on the GPU (using CUDA C++ in the backend). 

**Why use it?**
* **Zero Friction:** If you know NumPy, you already know CuPy.
* **Speed:** It provides out-of-the-box GPU acceleration for array operations.
* **Ease of use:** You can often port CPU code to GPU simply by changing `import numpy as np` to `import cupy as cp`.

In [1]:
import numpy as np
import cupy as cp
import cupyx as cpx
import matplotlib.pyplot as plt

# Helper to display benchmark results concisely.
def print_benchmark(result):
    """Print benchmark result using wall-clock (cpu_times) for fair comparison."""
    avg_ms = result.cpu_times.mean() * 1000
    std_ms = result.cpu_times.std() * 1000
    print(f"{result.name}: {avg_ms:.3f} ms +/- {std_ms:.3f} ms")

### 1. Creating Arrays: CPU vs. GPU

Let's compare the performance of creating a large 3D array (approx. 100 MB in size) on the CPU versus the GPU.

We will use `np.ones()` for the CPU and `cp.ones()` for the GPU.


In [2]:
# CPU creation
print_benchmark(cpx.profiler.benchmark(np.ones, ((50, 500, 500),), n_repeat=10, n_warmup=1))

ones: 5.027 ms +/- 0.082 ms


In [3]:
# GPU creation
print_benchmark(cpx.profiler.benchmark(cp.ones, ((50, 500, 500),), n_repeat=10, n_warmup=1))

ones: 0.017 ms +/- 0.015 ms


We can see here that creating this array on the GPU is much faster than doing so on the CPU!

**About `cupyx.profiler.benchmark()`:**

We use CuPy's built-in `benchmark()` utility for timing GPU operations. This is important because GPU operations are **asynchronous** - when you call a CuPy function, the CPU places a task in the GPU's "to-do list" (stream) and immediately moves on without waiting.

The `benchmark()` function handles all the complexity of proper GPU timing for us:
- It automatically synchronizes GPU streams to get accurate measurements.
- It runs warm-up iterations to avoid cold-start overhead.
- It reports both CPU wall-clock times (`cpu_times`) and GPU kernel times (`gpu_times`). We use `cpu_times` for all comparisons because it measures end-to-end wall-clock time, giving a fair apples-to-apples comparison between CPU and GPU code.

This makes it the recommended way to time CuPy code, as it's both accurate and convenient.

### 2. Basic Operations

The syntax for mathematical operations is identical. Let's multiply every value in our arrays by `5`.

In [4]:
multiply_shape = (50, 500, 500)

def multiply(x):
    return x * 5

In [5]:
# CPU Operation
x_cpu = np.ones(multiply_shape)
print_benchmark(cpx.profiler.benchmark(multiply, (x_cpu,), n_repeat=10, n_warmup=1))

multiply: 7.570 ms +/- 0.060 ms


In [6]:
# GPU Operation
x_gpu = cp.ones(multiply_shape)
print_benchmark(cpx.profiler.benchmark(multiply, (x_gpu,), n_repeat=10, n_warmup=1))

multiply: 0.025 ms +/- 0.018 ms


The GPU completes this operation notably faster, with the code staying the same.

#### Sequential Operations & Memory

Now let's do a couple of operations sequentially.

In [7]:
sequential_math_shape = (50, 500, 500)

def sequential_math(x):
    x = x * 5
    x = x * x
    x = x + x
    return x

Remember, each of these operations will return a **Copy**, not a **View**. This can be a common performance pitfall with NumPy and CuPy!

In [8]:
# CPU: Sequential math
x_cpu = np.ones(sequential_math_shape)
print_benchmark(cpx.profiler.benchmark(sequential_math, (x_cpu,), n_repeat=10, n_warmup=1))

sequential_math: 23.513 ms +/- 0.042 ms


In [9]:
# GPU: Sequential math
x_gpu = cp.ones(sequential_math_shape)
print_benchmark(cpx.profiler.benchmark(sequential_math, (x_gpu,), n_repeat=10, n_warmup=1))

sequential_math: 0.043 ms +/- 0.021 ms


But even with the copies, the GPU ran that much faster. The copies stay on the GPU; CuPy only transfers data from GPU to CPU when necessary or explicitly requested.

### 3. Complex Operations (Linear Algebra)

GPUs excel at Linear Algebra. Let's look at **Singular Value Decomposition (SVD)**, a computationally heavy $O(N^3)$ operation.

In [10]:
svd_shape = (3000, 1000)

In [11]:
# CPU SVD
x_cpu = np.random.random(svd_shape)
print_benchmark(cpx.profiler.benchmark(np.linalg.svd, (x_cpu,), n_repeat=5, n_warmup=1))

svd: 311.209 ms +/- 1.637 ms


In [12]:
# GPU SVD
x_gpu = cp.random.random(svd_shape)
print_benchmark(cpx.profiler.benchmark(cp.linalg.svd, (x_gpu,), n_repeat=5, n_warmup=1))

svd: 145.025 ms +/- 0.700 ms


The GPU outperforms the CPU again with exactly the same API!

#### Agnostic Code (NumPy Dispatch)

A key feature of CuPy is that many **NumPy functions work on CuPy arrays without changing your code**.

When you pass a CuPy GPU array (`x_gpu`) into a NumPy function that supports the `__array_function__` protocol (e.g., `np.linalg.svd()`), NumPy detects the CuPy input and **delegates the operation to CuPy’s own implementation**, which runs on the GPU.

This allows you to write code using standard `np.*` syntax and have it run on either CPU or GPU seamlessly - **as long as CuPy implements an override for that function.**

One common source of hidden performance penalties is **implicit transfers between CPU and GPU**. In some cases, CuPy guards against this: for example, when NumPy tries to convert a `cupy.ndarray` into a `numpy.ndarray` via the `__array__` protocol (e.g. `np.asarray(gpu_array)`), CuPy raises a `TypeError` instead of silently copying data to the host. 

However, CuPy **does** perform implicit GPU → CPU transfers in other cases, such as printing a GPU array, converting to a Python scalar (e.g. `float`, `.item()`), or evaluating a GPU scalar in a boolean context. We will explore these implicit transfers in a later notebook.

In [ ]:
# We create the data on the GPU...
x_gpu = cp.random.random(svd_shape)

# BUT we call the standard NumPy function - CuPy dispatches it to the GPU!
print_benchmark(cpx.profiler.benchmark(np.linalg.svd, (x_gpu,), n_repeat=5, n_warmup=1))

### 4. Device Management

If you have multiple GPUs, you can use a `with` statement to ensure specific arrays are created on specific devices (e.g., GPU 0 vs GPU 1).

In [13]:
with cp.cuda.Device(0):
   x_on_gpu0 = cp.random.random((100000, 1000))

print(f"Array is on device: {x_on_gpu0.device}")

Array is on device: <CUDA Device 0>


**Note:** CuPy functions generally expect all input arrays to be on the **same** device. Passing an array stored on a non-current device may work depending on the hardware configuration but is generally discouraged as it may not be performant.


---

### Exercise - NumPy to CuPy

#### Part 1
Let's put the "Drop-in Replacement" philosophy to the test with the same data pipeline as the previous notebook. Specifically, the single block of code below performs the following steps:
1) Generate a massive dataset (50 million elements).
2) Process it using a heavy operation (Sorting).
3) Manipulate the shape and normalize the data (Broadcasting).
4) Verify the integrity of the result.

**TODO:**
1. Run the cell below with `xp = np` (CPU Mode). Note the benchmark output.
2. Change the setup line to `xp = cp` (GPU Mode). Run it again.
3. Observe how the exact same logic runs significantly faster on the GPU with CuPy while retaining the implementation properties of NumPy.

Note: We use `cupyx.profiler.benchmark()` for timing, which automatically handles GPU synchronization.

In [17]:
# Step 1.) Setup: Choose Your Target
xp = cp  # Toggle this to 'cp' for GPU acceleration

print(f"Running on: {xp.__name__.upper()}")

# Step 2.) Data Generation
N = 50_000_000
print(f"Generating {N:,} random elements ({N*8/2**30:.2f} GB)...")
arr = xp.random.rand(N)

# Step 3.) Heavy Computation (Timed)
print("Sorting data...")
# cpx.profiler.benchmark() handles GPU synchronization automatically
result = cpx.profiler.benchmark(xp.sort, (arr,), n_repeat=5, n_warmup=1)
print_benchmark(result)

# Step 4.) Manipulation & Broadcasting
# Purpose: Demonstrate that CuPy supports complex reshaping and broadcasting rules exactly like NumPy.
# This shows you don't need to rewrite your data processing logic.

# Reshape to a matrix with 5 columns
arr_new = arr.reshape((-1, 5))

# Normalize: Divide every row by its sum using broadcasting
row_sums = arr_new.sum(axis=1)
normalized_matrix = arr_new / row_sums[:, xp.newaxis]

# Step 5.) Verification
# Purpose: Verify mathematical correctness/integrity of the result.
check_sums = xp.sum(normalized_matrix, axis=1)
xp.testing.assert_allclose(check_sums, 1.0)

print("Verification: PASSED (All rows sum to 1.0)")

Running on: CUPY
Generating 50,000,000 random elements (0.37 GB)...
Sorting data...
sort: 3.497 ms +/- 0.031 ms
Verification: PASSED (All rows sum to 1.0)


Numpy:

Running on: NUMPY
Generating 50,000,000 random elements (0.37 GB)...
Sorting data...
sort: 1186.199 ms +/- 2.290 ms
Verification: PASSED (All rows sum to 1.0)

CuPy:

Running on: CUPY
Generating 50,000,000 random elements (0.37 GB)...
Sorting data...
sort: 3.497 ms +/- 0.031 ms
Verification: PASSED (All rows sum to 1.0)

**TODO: When working with CuPy arrays, try changing `xp.testing.assert_allclose()` to `np.testing.assert_allclose()`. What happens and why?**

#### Part 2
We will now create a massive dataset (50 million points) representing a sine wave and see how fast the GPU can sort it compared to the CPU. 

**TODO:** 
1) **Generate Data:** Create a NumPy array (`y_cpu`) and a CuPy array (`y_gpu`) representing $\sin(x)$ from $0$ to $2\pi$ with `50,000,000` points.
2) **Benchmark CPU and GPU:** Use `benchmark()` from `cupyx.profiler` to measure both `np.sort()` and `cp.sort()`.

In [19]:
# Step 1.) Generate Data
N = 50_000_000
print(f"Generating {N} points...")

# TODO: Create x_cpu using np.linspace from 0 to 2*pi
x_cpu = np.linspace(0, 2 * np.pi, N)
# TODO: Create y_cpu by taking np.sin(x_cpu)
y_cpu = np.sin(x_cpu)

# TODO: Create x_gpu using cp.linspace from 0 to 2*pi
x_gpu = cp.linspace(0, 2 * cp.pi, N)
# TODO: Create y_gpu by taking cp.sin(x_gpu)
y_gpu = cp.sin(x_gpu)


# Step 2.) Benchmark NumPy (CPU)
print("Benchmarking NumPy Sort (this may take a few seconds)...")
# TODO: Use cpx.profiler.benchmark(function, (args,), n_repeat=5, n_warmup=1)
# Hint: Pass the function `np.sort()` and the argument `(y_cpu,)`
# Note: The comma in (y_cpu,) is required to make it a tuple!
result = cpx.profiler.benchmark(np.sort, (y_cpu,), n_repeat=5, n_warmup=1)
print_benchmark(result)


# Step 3.) Benchmark CuPy (GPU)
print("Benchmarking CuPy Sort...")
# TODO: Use cpx.profiler.benchmark(function, (args,), n_repeat=5, n_warmup=1)
# Hint: Pass the function `cp.sort()` and the argument `(y_gpu,)`
# Note: The comma in (y_gpu,) is required to make it a tuple!
result = cpx.profiler.benchmark(cp.sort, (y_gpu,), n_repeat=5, n_warmup=1)
print_benchmark(result)

Generating 50000000 points...
Benchmarking NumPy Sort (this may take a few seconds)...
sort: 1113.844 ms +/- 6.688 ms
Benchmarking CuPy Sort...
sort: 3.503 ms +/- 0.026 ms


**EXTRA CREDIT: Benchmark with different array sizes and find the size at which CuPy and NumPy take the same amount of time. Try to extract the timing data from `cupyx.profiler.benchmark()`'s return value and customize how the output is displayed. You could even make a graph.**

In [ ]:
sizes = [5, 50, 500, 5_000, 50_000, 500_000, 5_000_000, 50_000_000]

# TODO